# Realistic Level 0 multi-seed diagnostics

Aggregate completed AdamW or Muon runs using mean and one-standard-deviation bands. Test results are read from validation-selected checkpoints.


In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path(os.getenv("NANOGPT_LEVEL0_RESULTS_ROOT", "/tmp/nanogpt-level0-bpe/results"))
REPORT = ROOT / "multiseed-plots"
REPORT.mkdir(parents=True, exist_ok=True)
rows = []
selected_rows = []
for path in sorted(ROOT.glob("*_seed_*/metrics.csv")):
    optimizer, seed_text = path.parent.name.split("_seed_")
    frame = pd.read_csv(path)
    frame["optimizer"] = optimizer
    frame["seed"] = int(seed_text)
    rows.append(frame)
    selected_path = path.parent / "selected_checkpoint_metrics.json"
    if selected_path.exists():
        selected = json.loads(selected_path.read_text())
        selected.update({"optimizer": optimizer, "seed": int(seed_text)})
        selected_rows.append(selected)
if not rows:
    raise RuntimeError(f"No completed metrics found under {ROOT}")
all_metrics = pd.concat(rows, ignore_index=True)
selected_metrics = pd.DataFrame(selected_rows)
all_metrics.groupby("optimizer").seed.nunique()


In [ ]:
def band_plot(metric, ylabel=None):
    plt.figure(figsize=(10, 6))
    for optimizer, data in all_metrics.groupby("optimizer"):
        summary = data.groupby("step")[metric].agg(["mean", "std"]).reset_index()
        spread = summary["std"].fillna(0)
        line, = plt.plot(summary.step, summary["mean"], label=optimizer)
        plt.fill_between(summary.step, summary["mean"] - spread, summary["mean"] + spread, alpha=0.2, color=line.get_color())
    plt.xlabel("optimizer step")
    plt.ylabel(ylabel or metric)
    plt.title(f"{metric}: mean ± one standard deviation")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(REPORT / f"{metric}.png", dpi=180, bbox_inches="tight")
    plt.show()

for metric in ["val_loss", "val_perplexity", "val_accuracy", "val_generalization_gap"]:
    band_plot(metric)


In [ ]:
if not selected_metrics.empty:
    display(selected_metrics[["optimizer", "seed", "selected_step", "validation_loss", "test_loss", "test_perplexity", "test_accuracy"]].sort_values(["optimizer", "seed"]))
    summary = selected_metrics.groupby("optimizer")[["test_loss", "test_perplexity", "test_accuracy"]].agg(["mean", "std"])
    display(summary)
print(f"Saved plots to {REPORT}")
